In [11]:
# !pip install bs4
!pip install nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 19.5 MB/s eta 0:00:0000:0100:01


In [31]:
import os
import json
import random
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

from tqdm import tqdm

In [3]:
category = "Clothing, Shoes, and Jewellery"
# https://datarepo.eng.ucsd.edu/mcauley_group/data/amazon_v2/metaFiles2/meta_Clothing_Shoes_and_Jewelry.json.gz
products = 'Headdress, Boots, Jewellery, Wallet, Shirt, Dress, Underwear, Pants, Watch, Jacket, Sweater, Luggage'.split(', ')

In [22]:
url = 'https://cseweb.ucsd.edu/~jmcauley/datasets/amazon_v2/'

# Fetch the HTML content of the webpage
response = requests.get(url)

candidate_urls = []
if response.status_code == 200:
    # Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')
    # Find all links in the webpage
    links = soup.find_all('a', href=True)

    for link in links:
        # Construct the absolute URL using urljoin
        absolute_url = urljoin(url, link['href'])
        if 'Clothing_Shoes_and_Jewelry' in absolute_url and 'meta' in absolute_url:
            candidate_urls.append(absolute_url)

In [23]:
# def filename(url):
#     basename, _ = os.path.splitext(os.path.basename(url))
#     return os.path.splitext(basename)[0]

def download_gz_file(url, save_path):
    # Download gzipped content from the given URL
    response = requests.get(url)
    
    if response.status_code == 200:
        # Save the gzipped content to a file
        with open(save_path, 'wb') as file:
            file.write(response.content)
            print(f"Downloaded gzipped file from: {url} to: {save_path}")

for candidate_url in candidate_urls:
    download_gz_file(candidate_url, f'data/{os.path.basename(url)}')

KeyboardInterrupt: 

### Read data

In [41]:
def find_common_element(list1, list2):
    common_elements = [element for element in list2 if element in list1]
    if len(common_elements) == 1:
        return common_elements[0]
    return None

def read_lines_from_json_file(file_path, num_lines):
    with open(file_path, 'r') as file:
        lines = [json.loads(file.readline()) for _ in range(num_lines)]
    return lines

def check_url(url):
    response = requests.get(url)
    if response.status_code == 200:
        return True
    else:
        return False

# Example usage
file_path = 'data/meta_Clothing_Shoes_and_Jewelry.json'
num_lines_to_read = 1000
json_lines = read_lines_from_json_file(file_path, num_lines_to_read)

data = []
# Now you can work with the list of JSON data
for line in tqdm(json_lines):
    keys = line.keys()
    if 'imageURLHighRes' in keys and 'description' in keys and 'category' in keys and 'title' in keys and 300 > len(line['title']) > 30:
        url = random.choice(line['imageURLHighRes'])
        # if not check_url(url):
        #     continue
        category = find_common_element(line['category'], products)
        if category:
            data.append([category, url, line['title']])

100%|██████████| 1000/1000 [00:00<00:00, 392467.86it/s]


### Save Data

In [36]:
import csv

# Define the file path
csv_file_path = 'data.csv'

# Define the field names
field_names = ['category', 'imageURL', 'title']

# Write the data to the CSV file
with open(csv_file_path, 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(field_names)
    writer.writerows(data)

print(f"Data saved to {csv_file_path}")

Data saved to data.csv


In [40]:
import csv

# Define the file path
csv_file_path = 'data/amazon_products.csv'

# Create an empty list to store the data
data_list = []

# Read the data from the CSV file
with open(csv_file_path, 'r') as file:
    reader = csv.reader(file)
    next(reader)  # Skip the header row
    for row in reader:
        data_list.append(row)

# Print the data list
print(data_list)

Error: field larger than field limit (131072)

In [64]:
# import nltk
# nltk.download('punkt')  # Download the Punkt tokenizer models

# from nltk.tokenize import sent_tokenize

# def split_text_into_sentences(text):
#     sentences = sent_tokenize(text)
#     return sentences

# # Example usage
# text = data[0][2][0]
# sentences = split_text_into_sentences(text)

# # Print the result
# for i, sentence in enumerate(sentences, 1):
#     print(f"Sentence {i}: {sentence}")

Sentence 1: C


[nltk_data] Downloading package punkt to /nfs/home/dasaro/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [85]:
from PIL import Image
from io import BytesIO
import requests

def load_image_from_url(url):
    response = requests.get(url)
    
    if response.status_code == 200:
        image = Image.open(BytesIO(response.content))
        return image
    else:
        print(f"Failed to load image. Status code: {response.status_code}")
        return None

# Example usage
image_url = data[12][1]
loaded_image = load_image_from_url(image_url)

loaded_image.size

(455, 500)